Configure the local data snapshots as described in the README before running this notebook.

# Cross-market information and Ibovespa direction

This notebook reads an archived snapshot through 5 November 2026. It predicts the direction of the next Ibovespa close and evaluates BOVA11 as the execution instrument.

In [ ]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable,"-m","pip","-q","install","exchange-calendars","hmmlearn","arch","lightgbm","xgboost"])

In [ ]:
from local_snapshots import load_local_snapshot

SNAPSHOTS = load_local_snapshot('Cross_Market_Features_and_Meta_Labeling.ipynb', 'SNAPSHOTS')

# Intraday strategy snapshot, including causal state parameters.
FROZEN_INTRADAY = load_local_snapshot('Cross_Market_Features_and_Meta_Labeling.ipynb', 'FROZEN_INTRADAY')


## Prices and exchange calendar

BVMF sessions define the trading calendar. The barrier simulator excludes windows with incomplete execution prices or insufficient remaining sessions.

In [ ]:
from pathlib import Path
import io, base64, zlib, json
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import brier_score_loss, balanced_accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from hmmlearn.hmm import GaussianHMM
from arch import arch_model
import exchange_calendars as xcals

OUTPUT = Path("results_markets")/"P1_data"
OUTPUT.mkdir(parents=True,exist_ok=True)
SEED = 42
DATA_CAL = pd.Timestamp("2025-01-01")
DATA_VALID = pd.Timestamp("2025-07-01")
DATA_TEST = pd.Timestamp("2026-01-01")
COST_PER_SIDE = .0015  # 10 bps in costs plus 5 bps in slippage per side
HOLDING_PERIOD = 5
EXPLORATORY_END = pd.Timestamp("2026-11-05")
# Optional directory of additional Date/OHLC CSVs named after the snapshot assets.
# Append dates strictly after the last observation of each included series.
OUTPUT_DIR_DATA_NEW = None

def load(name):
    raw = zlib.decompress(base64.b64decode(SNAPSHOTS[name]))
    data = pd.read_csv(io.BytesIO(raw), index_col="Date", parse_dates=True)
    if OUTPUT_DIR_DATA_NEW is not None:
        file = Path(OUTPUT_DIR_DATA_NEW)/f"{name}.csv"
        if file.exists():
            new = pd.read_csv(file,index_col="Date",parse_dates=True)
            data = pd.concat([data,new.loc[new.index>data.index.max()]]).sort_index()
    data.index = data.index.astype("datetime64[ns]")
    assert data.index.is_unique and data.index.is_monotonic_increasing
    return data

ibov = load("ibov")
cal = xcals.get_calendar("BVMF", start="2020-11-01", end=ibov.index.max()+pd.Timedelta(days=15))
sessions = cal.sessions_in_range(ibov.index.min(), ibov.index.max())
assert sessions.equals(ibov.index), "The Ibovespa history contains gaps."
bova = load("bova11").reindex(sessions)
QUARANTINE_BOVA11 = {'2026-10-26': {'reason': 'OHLC differs across providers. The official opening price is unconfirmed.', 'open_yahoo': 140.0, 'open_alternativa': 169.0, 'maximum_alternativa': 176.6, 'minimum_alternativa': 165.06, 'close_alternativo': 172.42, 'source_alternativa': 'https://www.cotab3.com.br/cotacoes?months=12&ticker=BOVA11', 'reference_oficial': 'https://www.b3.com.br/pt_br/market-data-e-indexes/servicos-de-data/market-data/historical/mercado-a-vista/cotacoes-historicas/'}}
bova_execution = bova.copy()
em_quarantine = bova_execution.index.isin(pd.to_datetime(list(QUARANTINE_BOVA11)))
bova_execution.loc[em_quarantine,["Open","High","Low","Close"]] = np.nan
print("Sessions excluded from execution:",list(bova_execution.index[em_quarantine].date))
for data in [ibov, bova]:
    ohlc = data[["Open","High","Low","Close"]].dropna()
    assert np.isfinite(ohlc).all().all() and (ohlc > 0).all().all()
    assert (ohlc.High >= ohlc.max(axis=1)).all() and (ohlc.Low <= ohlc.min(axis=1)).all()
print("Ibovespa sessions:", len(ibov), "| BOVA11 sessions without complete OHLC:",
      bova[["Open","High","Low","Close"]].isna().any(axis=1).sum())

## Features available at the forecast origin

The features include momentum, volatility, regimes, and lagged returns for the S&P 500, USD/BRL, VIX, crude oil, and EWZ.

In [ ]:
# The fallback mirrors the module for standalone use in Colab.
try:
    from intraday_features import build_features
except ModuleNotFoundError as error:
    if error.name != "intraday_features":
        raise
    def build_features(ibov, sessions, load, frozen):
        """Build features for the frozen model in chronological order."""
        r = np.log(ibov.Close).diff()
        X = pd.DataFrame(index=sessions)
        for k in [0,1,2,3,4]:
            X[f"ret_lag{k}"] = r.shift(k)
        for w in [5,10,21,63]:
            X[f"ret_{w}"] = np.log(ibov.Close).diff(w)
            X[f"vol_{w}"] = r.rolling(w).std()
            X[f"dist_mean_{w}"] = np.log(ibov.Close/ibov.Close.rolling(w).mean())
        X["gap"] = np.log(ibov.Open/ibov.Close.shift())
        X["intraday"] = np.log(ibov.Close/ibov.Open)
        X["range"] = np.log(ibov.High/ibov.Low)
        X["day"] = sessions.day_of_week
        local = list(X.columns)

        # Parameters fitted on the initial 252 returns and stored in the snapshot.
        state_hmm = frozen["hmm"]
        mu = np.asarray(state_hmm["mean"])
        var = np.asarray(state_hmm["variance"])
        prior = np.asarray(state_hmm["initial"])
        transition = np.asarray(state_hmm["transition"])
        regime = []
        for obs in r.dropna().to_numpy()*100:
            logw = np.log(np.maximum(prior,1e-300))-.5*(np.log(var)+(obs-mu)**2/var)
            prior = np.exp(logw-logsumexp(logw)) @ transition
            regime.append(prior[np.argmax(var)])
        X["hmm_high_volatility"] = pd.Series(regime,index=r.dropna().index)
        state_garch = frozen["garch"]
        omega, alpha, beta = (state_garch[k] for k in ["omega","alpha","beta"])
        sigma2 = state_garch["variance_initial"]
        vars_predicted = []
        for obs in r.dropna().to_numpy()*100:
            sigma2 = omega + alpha*obs**2 + beta*sigma2
            vars_predicted.append(sigma2)
        X["garch_next_volatility"] = pd.Series(np.sqrt(vars_predicted)/100,index=r.dropna().index)
        dates_sources = pd.DataFrame(index=sessions)

        for name in ["sp500","usd","vix","oil","ewz"]:
            price = load(name).Close.dropna()
            if not np.isfinite(price).all() or not price.gt(0).all():
                raise ValueError(f"Invalid closing prices for {name}.")
            ext = pd.DataFrame({f"{name}_ret{w}":np.log(price).diff(w) for w in [1,5,21]})
            ext["available"] = ext.index
            aligned = pd.merge_asof(pd.DataFrame(index=sessions),ext,left_index=True,right_index=True,
                                     allow_exact_matches=False,tolerance=pd.Timedelta(days=7))
            if not (aligned.available.dropna() < aligned.available.dropna().index).all():
                raise ValueError(f"Future data availability for {name}.")
            dates_sources[name] = aligned.available
            X = X.join(aligned.drop(columns="available"))

        return r, X, local, dates_sources

r, X, local, dates_sources = build_features(ibov, sessions, load, FROZEN_INTRADAY)
target = r.shift(-1)
dates_target = pd.Series(sessions,index=sessions).shift(-1)
X_origins = X.iloc[253:].dropna()
X = X_origins.loc[target.reindex(X_origins.index).notna()]
y = target.reindex(X.index).gt(0).astype(int)
dates_target = dates_target.reindex(X.index)
assert len(X.loc[X.index>=DATA_TEST]) > 150

def period(start, end):
    return X.index[(X.index>=start) & (dates_target<end)]

train = period(X.index.min(), DATA_CAL)
calibration = period(DATA_CAL, DATA_VALID)
validation = period(DATA_VALID, DATA_TEST)
test = X.index[(X.index>=DATA_TEST) & (dates_target<=EXPLORATORY_END)]
assert dates_target.loc[train].max() < calibration.min()
assert dates_target.loc[calibration].max() < validation.min()
assert dates_target.loc[validation].max() < test.min()
print("training, calibration, validation, and test:", *(len(p) for p in [train,calibration,validation,test]))

## Daily classifier and calibration

Four candidates are compared across four chronological validation windows before 2025. The selective threshold requires at least 20% validation coverage. Test coverage is reported separately.

In [ ]:
def logistic():
    return make_pipeline(StandardScaler(),LogisticRegression(C=.05,random_state=SEED))

candidates = {
    "OHLC logistic regression":(logistic(),local),
    "Cross-market logistic regression":(logistic(),list(X.columns)),
    "XGBoost markets":(XGBClassifier(n_estimators=140,max_depth=2,learning_rate=.025,
        min_child_weight=12,subsample=.8,colsample_bytree=.8,random_state=SEED,n_jobs=2),list(X.columns)),
    "LightGBM markets":(LGBMClassifier(n_estimators=140,num_leaves=5,min_child_samples=45,
        learning_rate=.025,verbosity=-1,random_state=SEED,n_jobs=2),list(X.columns)),
}
oof = {}
adjusted = {}
for name,(model,cols) in candidates.items():
    probs = []
    for tr,va in TimeSeriesSplit(n_splits=4,test_size=120,gap=HOLDING_PERIOD).split(train):
        assert dates_target.loc[train[tr]].max() < train[va].min()
        fit = clone(model).fit(X.loc[train[tr],cols],y.loc[train[tr]])
        probs.append(pd.Series(fit.predict_proba(X.loc[train[va],cols])[:,1],index=train[va]))
    oof[name] = pd.concat(probs)
    adjusted[name] = clone(model).fit(X.loc[train,cols],y.loc[train])

def prob(name, dates):
    return adjusted[name].predict_proba(X.loc[dates,candidates[name][1]])[:,1]

def logits(p):
    p = np.clip(p,1e-5,1-1e-5)
    return np.log(p/(1-p)).reshape(-1,1)

calibradores = {name:LogisticRegression(C=1).fit(logits(prob(name,calibration)),y.loc[calibration])
               for name in candidates}
def prob_cal(name,dates):
    return calibradores[name].predict_proba(logits(prob(name,dates)))[:,1]

ranking = pd.DataFrame([
    {"model":name,"brier_oof":brier_score_loss(y.loc[p.index],p),
     "acuracia_oof":(y.loc[p.index].to_numpy()==(p.to_numpy()>=.5)).mean()}
    for name,p in oof.items()]).sort_values("brier_oof")
principal = ranking.iloc[0].model
p_val = prob_cal(principal,validation)
rules = []
for threshold in [.50,.525,.55,.575,.60,.65,.70]:
    asset = (p_val>=threshold)|(p_val<=1-threshold)
    if asset.sum()>=max(20,.20*len(validation)):
        rules.append(((y.loc[validation].to_numpy()[asset]==(p_val[asset]>=.5)).mean(),asset.sum(),threshold))
threshold_direction = max(rules,key=lambda row:row[:2])[2]
display(ranking)
print("Primary model:", principal, "| threshold:", threshold_direction)

## BOVA11 and meta-labeling

The primary classifier sets the trade direction. The secondary filter learns from historical signals generated outside each fitting fold. Simulated entry is at the next open, with a 1.5% profit target, a 1% stop, and a maximum of five sessions.

In [ ]:
try:
    from intraday_barriers import evaluate_events, uncensored_barrier
except ModuleNotFoundError as error:
    if error.name!="intraday_barriers":
        raise
    OHLC_BARRIERS = ["Open","High","Low","Close"]


    def simulate_event(bars, sessions, origin, side, limit, holding_period=5, cost_per_side=.0015):
        """Read prices in order after reserving a full window before the cutoff."""
        if side not in (-1,1) or holding_period<1 or not np.isfinite(cost_per_side) or cost_per_side<0:
            raise ValueError("Invalid trade side, holding period, or cost.")
        start = sessions.get_loc(origin)+1
        window = sessions[start:start+holding_period]
        event = {"origin":origin,"side":side,
                  "entry_date":window[0] if len(window) else pd.NaT,
                  "expiry":window[-1] if len(window)==holding_period else pd.NaT}
        if len(window)!=holding_period or window[-1]>=limit:
            return event|{"status":"outside_window","reason":"holding_period_crosses_cutoff"}
        marks = {}
        for pos,data in enumerate(window):
            if data not in bars.index:
                values = np.full(4,np.nan)
            else:
                values = bars.loc[data,OHLC_BARRIERS].to_numpy(dtype=float)
            open,maximum,minimum,close = values
            if (not np.isfinite(values).all() or (values<=0).any() or
                maximum<values.max() or minimum>values.min()):
                return event|{"status":"censored" if pos else "entry_unavailable",
                               "reason":"missing_or_invalid_ohlc","interruption_date":data}
            if pos==0:
                entry = open
                target = entry*(1+side*.015)
                stop = entry*(1-side*.01)
            worst = minimum if side==1 else maximum
            best = maximum if side==1 else minimum
            if side*(open-stop)<=0:
                price,reason = open,"stop_open"
            elif side*(open-target)>=0:
                price,reason = target,"target_open"
            elif side*(worst-stop)<=0:
                price,reason = stop,"stop"
            elif side*(best-target)>=0:
                price,reason = target,"target"
            elif pos==holding_period-1:
                price,reason = close,"holding_period"
            else:
                marks[data] = side*(close/entry-1)-cost_per_side
                continue
            net = side*(price/entry-1)-2*cost_per_side
            marks[data] = net
            return event|{"status":"closed","reason":reason,"output":data,
                           "net":net,"marks":marks}
        raise RuntimeError("An eligible window must end with an exit or censoring.")


    def evaluate_events(bars,sessions,probabilities,limit,holding_period=5,cost_per_side=.0015):
        """Return observed labels and an audit of every opportunity."""
        if not probabilities.index.is_unique or not np.isfinite(probabilities).all():
            raise ValueError("Opportunities require unique origins and finite values.")
        closed, audit = {}, []
        for origin,value in probabilities.sort_index().items():
            event = simulate_event(bars,sessions,origin,1 if value>=.5 else -1,
                                    limit,holding_period,cost_per_side)
            if event["status"]=="closed":
                closed[origin] = event
            audit.append({k:v for k,v in event.items() if k!="marks"})
        return closed,pd.DataFrame(audit,columns=["origin","side","entry_date",
            "expiry","status","reason","output","net","interruption_date"])


    def uncensored_barrier(bars, index, limit):
        """End the panel before the first session with unreliable OHLC data."""
        ohlc = bars.reindex(index)[OHLC_BARRIERS]
        valid = (np.isfinite(ohlc).all(axis=1)&ohlc.gt(0).all(axis=1)&
                  ohlc.High.ge(ohlc.max(axis=1))&ohlc.Low.le(ohlc.min(axis=1)))
        missing = ohlc.index[~valid]
        return min(limit,missing[0]) if len(missing) else limit

sessions_barriers = cal.sessions_in_range(sessions[0],sessions[-1]+pd.Timedelta(days=10))

def events(p,limit,label):
    records,audit = evaluate_events(bova_execution,sessions_barriers,p,limit,HOLDING_PERIOD,COST_PER_SIDE)
    audit.assign(threshold_exclusivo=limit).to_csv(
        OUTPUT/f"audit_barriers_{label}.csv",index=False)
    return records

def features_meta(p, records):
    idx = pd.DatetimeIndex(list(records))
    x = X.loc[idx].copy()
    x["side"] = np.where(p.loc[idx]>=.5,1,-1)
    x["forca"] = (p.loc[idx]-.5).abs()
    return x

p_oof = oof[principal]
ev_train = events(p_oof,DATA_CAL,"direcional_train")
xm = features_meta(p_oof,ev_train)
ym = [int(e["net"]>0) for e in ev_train.values()]
meta = logistic().fit(xm,ym)

def simulate(records, accepted, index):
    if len(index)==0:
        return {"trades":0,"hit":np.nan,"net_return":np.nan,
                "profit_factor":np.nan,"drawdown":np.nan,"sharpe":np.nan},pd.Series(dtype=float)
    equity = 1.
    last_output = pd.Timestamp.min
    curve = pd.Series(np.nan,index=index,dtype=float)
    trades = []
    for data,e in records.items():
        if data < last_output or not accepted.loc[data]:
            continue
        for day,marca in e["marks"].items():
            curve.loc[day] = equity*(1+marca)
        equity *= 1+e["net"]
        last_output = e["output"]
        trades.append(e["net"])
    curve = curve.ffill().fillna(1.)
    gains = sum(v for v in trades if v>0)
    losses = -sum(v for v in trades if v<0)
    rd = curve.pct_change().fillna(curve.iloc[0]-1)
    peak = curve.cummax().clip(lower=1.)
    return {"trades":len(trades),"hit":np.mean(np.array(trades)>0) if trades else np.nan,
            "net_return":equity-1,"profit_factor":gains/losses if losses else np.nan,
            "drawdown":(curve/peak-1).min(),
            "sharpe":np.sqrt(252)*rd.mean()/rd.std() if rd.std()>0 else np.nan},curve

# Train the filter on raw scores from the model family that generated the out-of-fold signals.
p_meta_val = pd.Series(prob(principal,validation),index=validation)
cal_val = sessions[(sessions>=DATA_VALID)&(sessions<DATA_TEST)]
end_val_barriers = uncensored_barrier(bova_execution,cal_val,DATA_TEST)
cal_val_barriers = cal_val[cal_val<end_val_barriers]
ev_val = events(p_meta_val,end_val_barriers,"direcional_validation")
xv = features_meta(p_meta_val,ev_val)
mv = pd.Series(meta.predict_proba(xv)[:,1],index=xv.index)
options = []
for threshold in [.45,.50,.55,.60,.65,.70]:
    met,_ = simulate(ev_val,mv>=threshold,cal_val_barriers)
    if met["trades"]>=12:
        options.append((met["net_return"],-threshold))
threshold_meta = -max(options)[1] if options else None
(OUTPUT/"rules_frozen.json").write_text(json.dumps({
    "principal":principal,"threshold_direction":float(threshold_direction),"threshold_meta":threshold_meta,
    "end_selection":"2025-12-31","cost_per_side":COST_PER_SIDE},indent=2))
print("Meta-model threshold:",threshold_meta)

## Directional evaluation in 2026

The tables report accuracy, coverage, and a majority-class benchmark on the same dates.

In [ ]:
p = prob_cal(principal,test)
real = y.loc[test].to_numpy()
maioria = int(y.loc[train].mean()>=.5)
rows = []
comparisons = [
    ("Majority class",np.full(len(test),maioria),np.ones(len(test),bool)),
    ("persistence",r.loc[test].gt(0).to_numpy(),np.ones(len(test),bool)),
    (principal+" seletivo",p>=.5,(p>=threshold_direction)|(p<=1-threshold_direction))]
comparisons.extend((name,prob_cal(name,test)>=.5,np.ones(len(test),bool))
                   for name in candidates)
for name, pred, asset in comparisons:
    rows.append({"model":name,"hits":int((pred[asset]==real[asset]).sum()),
        "signals":int(asset.sum()),"coverage":asset.mean(),
        "acuracia":(pred[asset]==real[asset]).mean() if asset.any() else np.nan,
        "acuracia_balanceada":balanced_accuracy_score(real[asset],pred[asset]) if len(np.unique(real[asset]))==2 else np.nan,
        "same_session_reference":(real[asset]==maioria).mean() if asset.any() else np.nan})
direction = pd.DataFrame(rows)
print("Test targets:",dates_target.loc[test].min().date(),"a",dates_target.loc[test].max().date())
print("Raw and calibrated Brier scores:", brier_score_loss(real,prob(principal,test)), brier_score_loss(real,p))
display(direction)
direction.to_csv(OUTPUT/"direction_2026.csv",index=False)
pd.DataFrame({"target_date":dates_target.loc[test],"high_real":real,"prob_high":p,
              "issued_signal":(p>=threshold_direction)|(p<=1-threshold_direction)}).to_csv(OUTPUT/"forecasts_2026.csv")

## Trading evaluation in 2026

Compare the primary signal with and without a filter under the same execution assumptions and costs. The cash position is a separate benchmark.

In [ ]:
p_meta = pd.Series(prob(principal,test),index=test)
idx_test = sessions[(sessions>=DATA_TEST)&(sessions<=EXPLORATORY_END)]
end_test_barriers = uncensored_barrier(bova_execution,idx_test,EXPLORATORY_END+pd.Timedelta(days=1))
idx_barriers = idx_test[idx_test<end_test_barriers]
ev_test = events(p_meta,end_test_barriers,"direcional_test")
xt = features_meta(p_meta,ev_test)
mt = pd.Series(meta.predict_proba(xt)[:,1] if len(xt) else [],index=xt.index,dtype=float)
filtros = {"Unfiltered":pd.Series(True,index=xt.index),"Caixa":pd.Series(False,index=xt.index)}
if threshold_meta is not None:
    filtros["Meta-labeling"] = mt>=threshold_meta
metrics = {}
curves = {}
for name,accepted in filtros.items():
    metrics[name],curves[name] = simulate(ev_test,accepted,idx_barriers)
trade = pd.DataFrame(metrics).T
trade["sessions_avaliados"] = len(idx_barriers)
display(trade)
trade.to_csv(OUTPUT/"trades_2026.csv")
pd.DataFrame(curves).to_csv(OUTPUT/"curves_2026.csv")
ax = pd.DataFrame(curves).plot(title="BOVA11 in 2026 after simulated costs",ylabel="Equity per unit of initial capital")
ax.figure.tight_layout()
ax.figure.savefig(OUTPUT/"curves_2026.png",dpi=150)

## Earlier version

Archived results preceding the unified barrier rules are retained in the original research workspace. The cells above produce the current historical evaluation.

## Persistence diagnostics

The directional forecast targets the next Ibovespa close. Barrier trades enter BOVA11 at the following open and use a maximum holding period of five sessions. The diagnostics break results down by the available groups.

In [ ]:
from arch.bootstrap import CircularBlockBootstrap
from xgboost import XGBRegressor

persistence = r.reindex(X.index).gt(0).astype(int)
lim_vol = float(X.loc[train,"garch_next_volatility"].median())
states = pd.DataFrame({
    "volatility": np.where(X.garch_next_volatility>=lim_vol,"high","low"),
    "regime_hmm": np.where(X.hmm_high_volatility>=.5,"high","low")},index=X.index)
z = X[["hmm_high_volatility","garch_next_volatility"]].copy()
z["direction_previous"] = 2*persistence-1
z["direction_x_regime"] = z.direction_previous*z.hmm_high_volatility
model_regime = logistic().fit(z.loc[train],y.loc[train])

ret_norm = target.reindex(X.index)/X.garch_next_volatility
def regressor():
    return XGBRegressor(n_estimators=140,max_depth=2,learning_rate=.025,min_child_weight=12,
                        subsample=.8,colsample_bytree=.8,random_state=SEED,n_jobs=2)
model_normalized = regressor().fit(X.loc[train],ret_norm.loc[train])

def diagnosticar_direction(dates, label):
    reais = y.loc[dates]
    forecasts = pd.DataFrame({
        "persistence":persistence.loc[dates],
        "Majority class":int(y.loc[train].mean()>=.5),
        "Previous model":(prob_cal(principal,dates)>=.5).astype(int),
        "HMM + persistence":model_regime.predict(z.loc[dates]),
        "XGBoost normalized return":(model_normalized.predict(X.loc[dates])>0).astype(int)
    },index=dates)
    hits = forecasts.eq(reais,axis=0)
    print(label, "|", len(dates), "forecasts")
    display(pd.DataFrame({"hits":hits.sum(),"acuracia":hits.mean()}))
    por_state = []
    for field in states:
        table = hits.assign(group=states.loc[dates,field]).groupby("group")
        contagens = table.size()
        result = table.mean().assign(days=contagens,segmentacao=field)
        por_state.append(result.reset_index())
    display(pd.concat(por_state,ignore_index=True))
    pd.concat(por_state,ignore_index=True).to_csv(OUTPUT/f"regimes_{label}.csv",index=False)
    rows = []
    if len(dates)>=40:
        for name in hits.columns:
            dif = hits[name].astype(float)-hits["persistence"].astype(float)
            for block in [5,10,20]:
                bs = CircularBlockBootstrap(block,dif.to_numpy(),seed=SEED)
                ic = bs.conf_int(np.mean,reps=3000,method="percentile")
                rows.append({"model":name,"block":block,"dif_pp":100*dif.mean(),
                               "IC_inf_pp":100*ic[0,0],"IC_sup_pp":100*ic[1,0]})
        intervals = pd.DataFrame(rows)
        display(intervals.loc[intervals.block.eq(10)])
        intervals.to_csv(OUTPUT/f"intervals_paired_{label}.csv",index=False)
    return hits

# Pre-2026 history provides descriptive results for the fixed persistence rule.
windows = {"validation_2025":validation,"exploratory_2026":test}
display(pd.DataFrame([{"window":name,"n":len(d),
    "acuracia_persistence":persistence.loc[d].eq(y.loc[d]).mean()} for name,d in windows.items()]))

## Persistence filter and expected net return

The filter uses the persistence direction and net barrier outcomes. The threshold is selected in the second half of 2025 with a minimum of 12 trades.

In [ ]:
def forecast_persistence(dates):
    return persistence.loc[dates].astype(float)

ev_persist_train = events(forecast_persistence(train),DATA_CAL,"persistence_train")
xp = features_meta(forecast_persistence(train),ev_persist_train)
model_filter_persist = logistic().fit(xp,[int(e["net"]>0) for e in ev_persist_train.values()])

def pontuar_persistence(dates, end, label):
    primario = forecast_persistence(dates)
    ev = events(primario,end,label)
    if not ev:
        return ev,pd.Series(dtype=float)
    xm = features_meta(primario,ev)
    return ev,pd.Series(model_filter_persist.predict_proba(xm)[:,1],index=xm.index)

ev_pv, pontuacao_pv = pontuar_persistence(validation,end_val_barriers,"persistence_validation")
choices = []
for threshold in [.45,.50,.55,.60,.65,.70]:
    met,_ = simulate(ev_pv,pontuacao_pv>=threshold,cal_val_barriers)
    if met["trades"]>=12:
        choices.append((met["net_return"],-threshold))
threshold_persist = -max(choices)[1] if choices else None

ret_bova = (bova_execution.Close/bova_execution.Open-1).shift(-1).reindex(X.index)
import hashlib
model_bytes = base64.b64decode(FROZEN_INTRADAY["model_ubj"])
assert hashlib.sha256(model_bytes).hexdigest() == FROZEN_INTRADAY["sha256"]
assert list(X.columns) == FROZEN_INTRADAY["features"]
assert COST_PER_SIDE == FROZEN_INTRADAY["cost_per_side"]
model_ret = XGBRegressor()
model_ret.load_model(bytearray(model_bytes))
margin = FROZEN_INTRADAY["margin"]
(OUTPUT/"model_intraday_frozen.ubj").write_bytes(model_bytes)
manifest = {k:v for k,v in FROZEN_INTRADAY.items() if k!="model_ubj"}
manifest["first_origin_confirmation"] = str((EXPLORATORY_END+pd.Timedelta(days=1)).date())
(OUTPUT/"manifest_intraday_frozen.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
(OUTPUT/"rules_diagnostic.json").write_text(json.dumps({
    "train_ate":"2024-12-31","validation_ate":"2025-12-31",
    "exploratory_end":str(EXPLORATORY_END.date()),"lim_vol_garch":lim_vol,
    "threshold_hmm":.5,"threshold_filter_persistence":threshold_persist,
    "cost_per_side":COST_PER_SIDE,"margin_return_value_expected":margin,
    "blocks_bootstrap":[5,10,20],"seed":SEED},indent=2),encoding="utf-8")

def medir_trades(records, accepted, index):
    executed = {}
    last = pd.Timestamp.min
    for origin,e in records.items():
        if origin>=last and accepted.loc[origin]:
            executed[origin] = e
            last = e["output"]
    met,curve = simulate(executed,pd.Series(True,index=list(executed)),index)
    net_returns = np.array([e["net"] for e in executed.values()])
    gains,losses = net_returns[net_returns>0],-net_returns[net_returns<0]
    met["hits_equilibrio"] = losses.mean()/(gains.mean()+losses.mean()) if len(gains) and len(losses) else np.nan
    met["exposure"] = sum(len(e["marks"]) for e in executed.values())/len(index) if len(index) else np.nan
    met["turnover_por_100_sessions"] = 200*len(executed)/len(index) if len(index) else np.nan
    return met,curve,executed

def evaluate_trades(dates, index, end, label):
    end_barriers = uncensored_barrier(bova_execution,index,end)
    index_barriers = index[index<end_barriers]
    ev, pontuacao = pontuar_persistence(dates,end_barriers,f"persistence_{label}")
    rules = {"Persistence with barriers":(ev,pd.Series(True,index=pontuacao.index))}
    if threshold_persist is not None:
        rules["persistence + meta-labeling"] = (ev,pontuacao>=threshold_persist)
    predicted = pd.Series(model_ret.predict(X.loc[dates].set_axis(model_ret.get_booster().feature_names or X.columns, axis=1))*X.loc[dates,"garch_next_volatility"],index=dates)
    intraday = {}
    for origin,v in predicted.items():
        data = dates_target.loc[origin]
        ret = ret_bova.loc[origin]
        if pd.notna(ret) and data<end and abs(v)>2*COST_PER_SIDE+margin:
            net = np.sign(v)*ret-2*COST_PER_SIDE
            intraday[origin] = {"output":data,"net":net,"marks":{data:net}}
    rules["Expected intraday return"] = (intraday,pd.Series(True,index=list(intraday)))
    metrics,curves,executed = {},{},{}
    for name,(rule_events,accepted) in rules.items():
        metrics[name],curves[name],executed[name] = medir_trades(
            rule_events,accepted,index if name=="Expected intraday return" else index_barriers)
        metrics[name]["sessions_avaliados"] = len(index if name=="Expected intraday return" else index_barriers)
    tab = pd.DataFrame(metrics).T
    display(tab)
    tab.to_csv(OUTPUT/f"trades_diagnostic_{label}.csv")
    pd.DataFrame(curves).to_csv(OUTPUT/f"curves_diagnostic_{label}.csv")
    pd.DataFrame(curves).plot(title=f"BOVA11 · {label}",ylabel="Equity per unit of initial capital")
    filtered = executed.get("persistence + meta-labeling",{})
    if filtered:
        # Retrospective control matched by duration without sampling on returns.
        base = list(executed["Persistence with barriers"].values())
        groups = {}
        for duration in range(1,HOLDING_PERIOD+1):
            n_target = sum(len(e["marks"])==duration for e in filtered.values())
            pool = np.array([e["net"] for e in base if len(e["marks"])==duration])
            if n_target:
                groups[duration] = (n_target,pool)
        if all(n<=len(pool) for n,pool in groups.values()):
            rng = np.random.default_rng(SEED)
            return_value_randomized = np.ones(2000)
            for n,pool in groups.values():
                return_value_randomized *= [np.prod(1+rng.choice(pool,size=n,replace=False))
                                     for _ in range(len(return_value_randomized))]
            q = np.quantile(return_value_randomized-1,[.025,.5,.975])
            control = pd.DataFrame([{"trades":len(filtered),"q025":q[0],"median":q[1],
                                      "q975":q[2],"return_value_filter":tab.loc["persistence + meta-labeling","net_return"]}])
            display(control)
            control.to_csv(OUTPUT/f"control_exposure_{label}.csv",index=False)
        else:
            print("The unfiltered baseline has too few trades of matching durations.")
    return tab

## Exploratory evaluation and future protocol

Confidence bins use calibrated class probabilities. Trading summaries include exposure, turnover, and the break-even hit rate. The historical exercise is exploratory, and prospective signals follow a separate protocol.

In [ ]:
hits_expl = diagnosticar_direction(test,"exploratory_2026")
prob_expl = prob_cal(principal,test)
confidence = np.maximum(prob_expl,1-prob_expl)
bins = pd.DataFrame({"bin":pd.cut(confidence,[.5,.55,.6,.65,.7,1.],include_lowest=True),
    "confidence":confidence,"hit":hits_expl["Previous model"].to_numpy()})
calib = bins.groupby("bin",observed=False).agg(
    signals=("hit","size"),confidence_mean=("confidence","mean"),acuracia=("hit","mean"))
calib["coverage"] = calib.signals/len(test)
display(calib)
calib.to_csv(OUTPUT/"calibration_bins_2026.csv")
trades_expl = evaluate_trades(test,idx_test,EXPLORATORY_END+pd.Timedelta(days=1),"exploratory_2026")
confirmation = X.index[X.index>EXPLORATORY_END]
if len(confirmation):
    hits_confirmation = diagnosticar_direction(confirmation,"confirmation")
    evaluate_trades(confirmation,sessions[sessions>EXPLORATORY_END],
                   sessions[-1]+pd.Timedelta(days=1),"confirmation")
else:
    print("The snapshot ends on November 5, 2026. Later evaluation requires additional sessions.")

## Earlier persistence diagnostics

An earlier version recorded 116 correct persistence predictions out of 210. The current analysis applies the unified barrier and execution rules.

## Audit of the frozen intraday signal

The local snapshot supplies the XGBoost model trained through 2024, its HMM and GARCH parameters, and the feature order. A hash identifies the model. The decision margin is 0.05% above estimated round-trip costs.

In [ ]:
from statsmodels.stats.proportion import proportion_confint

def audit_intraday(dates, label):
    origins = pd.DatetimeIndex(dates)
    executions = pd.DatetimeIndex(dates_target.loc[origins])
    assert executions.equals(sessions[sessions.get_indexer(origins)+1])
    sources = dates_sources.loc[origins].add_prefix("source_")
    for column in sources:
        assert sources[column].notna().all()
        assert (sources[column] < origins).all()
    bars = bova_execution.loc[executions,["Open","Close"]].set_axis(origins)
    forecast = model_ret.predict(X.loc[origins].set_axis(model_ret.get_booster().feature_names or X.columns, axis=1))*X.loc[origins,"garch_next_volatility"]
    eligible = bars.notna().all(axis=1)
    signal = forecast.abs()>2*COST_PER_SIDE+margin
    log = pd.DataFrame({"execution":executions,"forecast_raw":forecast,
        "eligible":eligible,"signal":signal,"open":bars.Open,"close":bars.Close,
        "volatility":states.loc[origins,"volatility"],
        "trend_21":np.where(X.loc[origins,"ret_21"]>=0,"Positive","Negative")}).join(sources)
    log.index.name = "origin"
    trades = log.loc[eligible&signal].copy()
    trades["side"] = np.sign(trades.forecast_raw).astype(int)
    trades["raw"] = trades.side*(trades.close/trades.open-1)
    trades["cost_total"] = 2*COST_PER_SIDE
    trades["net"] = trades.raw-trades.cost_total
    log.to_csv(OUTPUT/f"audit_origins_{label}.csv")
    trades.to_csv(OUTPUT/f"audit_trades_{label}.csv")
    if trades.empty:
        print(label, "| no trade with an observed outcome")
        return None
    n = len(trades)
    hits = int(trades.net.gt(0).sum())
    lo,hi = proportion_confint(hits,n,method="wilson")
    return_value = float((1+trades.net).prod()-1)
    mean = float(trades.net.mean())
    eligible_rows = log.loc[eligible]
    pool = (eligible_rows.close/eligible_rows.open-1).to_numpy()
    rng = np.random.default_rng(SEED)
    draws = np.array([rng.choice(len(pool),size=n,replace=False) for _ in range(10000)])
    pnl = pool[draws]*trades.side.to_numpy()-2*COST_PER_SIDE
    control = np.prod(1+pnl,axis=1)-1
    q = np.quantile(control,[.025,.5,.975])
    summary = pd.DataFrame([{"origins":len(log),"eligible":int(eligible.sum()),
        "signals_without_price":int((signal&~eligible).sum()),"trades":n,
        "long_positions":int(trades.side.eq(1).sum()),"short_positions":int(trades.side.eq(-1).sum()),
        "hits":hits,"precision":hits/n,"wilson_inf":lo,"wilson_sup":hi,
        "expectancy_net":mean,"net_return":return_value,
        "control_q025":q[0],"control_median":q[1],"control_q975":q[2],
        "percentile_vs_control":(control<=return_value).mean()}])
    display(summary.T)
    summary.to_csv(OUTPUT/f"audit_summary_{label}.csv",index=False)
    groups = []
    for column in ["volatility","trend_21"]:
        for group, candidates in log.loc[eligible].groupby(column):
            active = trades.loc[trades.index.intersection(candidates.index),"net"]
            groups.append({"axis":column,"group":group,"eligible":len(candidates),
                "trades":len(active),"hits":int(active.gt(0).sum()),
                "coverage":len(active)/len(candidates),"expectancy_net":active.mean()})
    segments = pd.DataFrame(groups)
    display(segments)
    segments.to_csv(OUTPUT/f"audit_regimes_{label}.csv",index=False)
    scenarios = []
    for slip_bps in [0,5,10,20]:
        net_returns = trades.raw-2*(10+slip_bps)/10000
        scenarios.append({"slippage_bps_per_side":slip_bps,"trades":n,
            "hits":int(net_returns.gt(0).sum()),"expectancy_net":net_returns.mean(),
            "net_return":(1+net_returns).prod()-1})
    sensitivity = pd.DataFrame(scenarios)
    display(sensitivity)
    sensitivity.to_csv(OUTPUT/f"audit_slippage_{label}.csv",index=False)
    return summary

historical_audit = audit_intraday(test,"exploratory_2026")
previous = trades_expl.loc["Expected intraday return"]
assert historical_audit.iloc[0].trades == previous.trades
assert np.isclose(historical_audit.iloc[0].net_return,previous.net_return)
if len(confirmation):
    audit_intraday(confirmation,"confirmation")
else:
    print("Confirmation requires data after November 5, 2026.")
